In [1]:
import os
import random
import warnings
import time
from pathlib import Path
from PIL import Image
import cv2
import numpy as np
from scipy.linalg import block_diag
from scipy.ndimage import gaussian_filter, sobel

from joblib import Parallel, delayed, cpu_count
from deap import base, creator, tools, gp
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# CẤU HÌNH HỆ THỐNG & MTGCN PARAMETERS
# ==========================================
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

print("[HỆ THỐNG] EXACT MTGCN Surrogate")

LIMIT = 1e9 
NUM_TREES = 15  
POP_SIZE = 100
MAX_GEN = 50
CXPB = 0.8   
MUTPB = 0.19 
ELITISM = int(0.01 * POP_SIZE) 

# Tham số Surrogate
P_RATIO = 0.2  
GNN_EPOCHS = 10
GNN_LR = 0.005
MAX_ARCHIVE_SIZE = 300 

THU_MUC_GOC = '/kaggle/input/datasets/xixuhu/office31/Office-31/webcam'
THU_MUC_MOI = '/kaggle/working/aberdeen_split_50x50'
DATA_DIR_TRAIN = os.path.join(THU_MUC_MOI, 'train')
DATA_DIR_TEST = os.path.join(THU_MUC_MOI, 'test')

# Sử dụng GPU cho PyTorch nếu có, ngược lại dùng CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[HỆ THỐNG] PyTorch đang sử dụng thiết bị: {DEVICE}")

# ==========================================
# 1. HÀM CHUẨN BỊ & TẢI DỮ LIỆU
# ==========================================
def prepare_train_test_dataset(src_dir, dest_dir, num_train=5, img_size=(50, 50)):
    duong_dan_goc = Path(src_dir)
    thu_muc_train = Path(dest_dir) / 'train'
    thu_muc_test = Path(dest_dir) / 'test'
    
    thu_muc_train.mkdir(parents=True, exist_ok=True)
    thu_muc_test.mkdir(parents=True, exist_ok=True)
    
    dinh_dang_anh = {'.jpg', '.jpeg', '.png', '.bmp'}
    tong_train = tong_test = 0
    
    print(f"Bắt đầu chia tập Train/Test (Train: {num_train} ảnh/class, Resize: {img_size})...", flush=True)
    
    for thu_muc_con in duong_dan_goc.iterdir():
        if not thu_muc_con.is_dir(): continue
        (thu_muc_train / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        (thu_muc_test / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        
        danh_sach_anh = [f for f in thu_muc_con.glob('*') if f.is_file() and f.suffix.lower() in dinh_dang_anh]
        random.shuffle(danh_sach_anh)
        
        so_luong_train = min(num_train, len(danh_sach_anh))
        anh_train = danh_sach_anh[:so_luong_train]
        anh_test = danh_sach_anh[so_luong_train:]
        
        def process_and_save(img_paths, dest_folder):
            count = 0
            for path in img_paths:
                try:
                    with Image.open(path) as img:
                        img = img.convert('RGB')
                        img_resized = img.resize(img_size, Image.Resampling.LANCZOS)
                        img_resized.save(dest_folder / path.name)
                        count += 1
                except Exception:
                    pass
            return count

        tong_train += process_and_save(anh_train, thu_muc_train / thu_muc_con.name)
        tong_test += process_and_save(anh_test, thu_muc_test / thu_muc_con.name)
        
    print(f"Hoàn thành! Tổng Train: {tong_train} ảnh | Tổng Test: {tong_test} ảnh\n")

def load_dataset_raw(data_dir):
    images, labels = [], []
    if not os.path.exists(data_dir):
        return np.array([]), np.array([])
    class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    class_mapping = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        for img_name in os.listdir(class_dir):
            img = cv2.imread(os.path.join(class_dir, img_name))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img / 255.0) 
                labels.append(class_mapping[class_name])
    return np.array(images), np.array(labels, dtype=np.int32)

# ==========================================
# 2. TOÁN TỬ GP & KHỞI TẠO DEAP
# ==========================================
def img_add(img1, img2): return np.clip(img1 + img2, 0.0, 1.0)
def img_sub(img1, img2): return np.clip(img1 - img2, 0.0, 1.0)
def img_max(img1, img2): return np.maximum(img1, img2)
def img_relu(img): return np.maximum(0, img)
def img_gaussian_blur(img):
    result = np.zeros_like(img)
    for c in range(3): result[:, :, c] = gaussian_filter(img[:, :, c], sigma=1.0)
    return result
def img_sobel_edges(img):
    result = np.zeros_like(img)
    for c in range(3):
        sx = sobel(img[:, :, c], axis=0)
        sy = sobel(img[:, :, c], axis=1)
        result[:, :, c] = np.hypot(sx, sy)
    return np.clip(result, 0.0, 1.0)

if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax, fold_accuracies=list, parent_fitness=list)

pset = gp.PrimitiveSet("MAIN", 1)
pset.renameArguments(ARG0="Image")
pset.addPrimitive(img_add, 2); pset.addPrimitive(img_sub, 2); pset.addPrimitive(img_max, 2)
pset.addPrimitive(img_gaussian_blur, 1); pset.addPrimitive(img_sobel_edges, 1); pset.addPrimitive(img_relu, 1)

toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)
toolbox.register("tree", tools.initIterate, gp.PrimitiveTree, toolbox.expr)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.tree, n=NUM_TREES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# ==========================================
# 3. CÁC HÀM ĐÁNH GIÁ THỰC TẾ (REAL FITNESS)
# ==========================================
def gp_transform_data(individual, X_data, pset_local, grid_size=(2, 2)):
    """
    Cải tiến: Spatial Block Pooling + Statistical Moments (Mean, Std, Max)
    """
    funcs = [gp.compile(expr=tree, pset=pset_local) for tree in individual]
    
    # Tính toán số lượng features
    # 3 kênh màu * 3 thống kê (mean, std, max) * số lượng blocks (vd: 2x2 = 4)
    features_per_tree = 3 * 3 * (grid_size[0] * grid_size[1])
    num_features = NUM_TREES * features_per_tree
    
    X_features = np.zeros((X_data.shape[0], num_features))
    
    for i in range(X_data.shape[0]):
        img_input = X_data[i]
        h, w, c = img_input.shape
        
        # Kích thước của từng khối (block)
        h_step, w_step = h // grid_size[0], w // grid_size[1]
        
        for j, func in enumerate(funcs):
            base_idx = j * features_per_tree
            try:
                processed_img = func(img_input)
                tree_feats = []
                
                # Trích xuất đặc trưng theo từng ô trong lưới 2x2
                for r in range(grid_size[0]):
                    for c_idx in range(grid_size[1]):
                        # Cắt lấy vùng ảnh con
                        block = processed_img[
                            r * h_step : (r + 1) * h_step,
                            c_idx * w_step : (c_idx + 1) * w_step,
                            :
                        ]
                        
                        # Trích xuất 3 loại thông số thống kê cho block này
                        block_mean = np.mean(block, axis=(0, 1))
                        block_std = np.std(block, axis=(0, 1))
                        block_max = np.max(block, axis=(0, 1))
                        
                        tree_feats.extend(block_mean)
                        tree_feats.extend(block_std)
                        tree_feats.extend(block_max)
                        
                # Cập nhật mảng đặc trưng, xử lý các lỗi nan/inf nếu có
                X_features[i, base_idx : base_idx + features_per_tree] = np.nan_to_num(
                    tree_feats, nan=0.0, posinf=1.0, neginf=0.0
                )
            except Exception:
                # Nếu cây GP bị lỗi toán học, gán toàn bộ đặc trưng của cây đó = 0
                X_features[i, base_idx : base_idx + features_per_tree] = 0.0
                
    return X_features

def evaluate_multitree_core(ind, X_data, y_data, pset_local):
    X_features = gp_transform_data(ind, X_data, pset_local)
    X_features = np.nan_to_num(X_features, nan=0.0, posinf=LIMIT, neginf=-LIMIT)
    
    clf = SVC(kernel='linear', max_iter=2000, random_state=42)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) 
    
    accuracies = []
    for train_index, test_index in skf.split(X_features, y_data):
        X_train, X_test = X_features[train_index], X_features[test_index]
        y_train, y_test = y_data[train_index], y_data[test_index]
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        try:
            clf.fit(X_train, y_train)
            accuracies.append(float(np.mean(clf.predict(X_test) == y_test)))
        except:
            accuracies.append(0.0)
    return np.mean(accuracies), accuracies

def evaluate_wrapper(args):
    ind, X_data, y_data, pset_local = args
    return evaluate_multitree_core(ind, X_data, y_data, pset_local)

def get_ind_hash(ind):
    return hash(str([str(tree) for tree in ind]))

# ==========================================
# 4. MODULE MTGCN SURROGATE (100% THEO BÀI BÁO GỐC)
# ==========================================
# Thêm 'None' để đại diện cho việc không có nút con
NODE_MAPPING = {
    'None': 0, 
    'Image': 1, 'img_add': 2, 'img_sub': 3, 'img_max': 4,
    'img_gaussian_blur': 5, 'img_sobel_edges': 6, 'img_relu': 7
}
NUM_NODE_TYPES = len(NODE_MAPPING)

class TempTreeNode:
    def __init__(self, idx, name):
        self.idx = idx
        self.name = name
        self.children = []
        self.height = 1
        self.descendants = 0

def compute_tree_metrics(node):
    desc = 0
    max_h = 0
    for child in node.children:
        c_desc, c_h = compute_tree_metrics(child)
        desc += 1 + c_desc
        if c_h > max_h:
            max_h = c_h
    node.descendants = desc
    node.height = max_h + 1
    return node.descendants, node.height

def extract_mtgcn_multitree_exact(individual):
    """
    Trích xuất đúng 5 đặc trưng: [Name, Left, Right, Height, Descendant]
    Hợp nhất 15 cây thành Đồ thị phân mảnh.
    """
    all_features = []  
    all_A_blocks = []  
    
    for tree in individual:
        n_nodes = len(tree)
        A = np.zeros((n_nodes, n_nodes), dtype=np.float32)
        
        nodes = []
        stack = []
        # Xây dựng lại cấu trúc cây từ list
        for i, deap_node in enumerate(tree):
            node_name = deap_node.name if isinstance(deap_node, gp.Primitive) else deap_node.value
            t_node = TempTreeNode(i, node_name)
            nodes.append(t_node)
            
            if stack:
                parent_t_node, arity_left = stack[-1]
                parent_t_node.children.append(t_node)
                A[parent_t_node.idx, i] = 1.0
                
                stack[-1] = (parent_t_node, arity_left - 1)
                if stack[-1][1] == 0:
                    stack.pop()
                    
            if isinstance(deap_node, gp.Primitive) and deap_node.arity > 0:
                stack.append((t_node, deap_node.arity))
                
        # Tính toán Độ cao và Số hậu duệ
        if nodes:
            compute_tree_metrics(nodes[0]) 
            
        # Trích xuất 5 đặc trưng
        for t_node in nodes:
            name_id = NODE_MAPPING.get(t_node.name, 0)
            left_id = NODE_MAPPING['None']
            right_id = NODE_MAPPING['None']
            
            if len(t_node.children) > 0:
                left_id = NODE_MAPPING.get(t_node.children[0].name, 0)
            if len(t_node.children) > 1:
                right_id = NODE_MAPPING.get(t_node.children[1].name, 0)
                
            height = min(t_node.height, 49) # Giới hạn không vượt quá embedding size
            desc = min(t_node.descendants, 499)
            
            all_features.append([name_id, left_id, right_id, height, desc])
            
        A = A + np.eye(n_nodes) 
        all_A_blocks.append(A)
        
    big_A = block_diag(*all_A_blocks)
    return np.array(all_features), big_A

class Exact_MTGCN_Model(nn.Module):
    def __init__(self, num_node_types, r=8, hidden_dim=32, max_height=50, max_desc=500):
        super().__init__()
        # 5 Lớp Embedding độc lập cho 5 cột đặc trưng
        self.emb_name = nn.Embedding(num_node_types, r)
        self.emb_left = nn.Embedding(num_node_types, r)
        self.emb_right = nn.Embedding(num_node_types, r)
        self.emb_height = nn.Embedding(max_height, r) 
        self.emb_desc = nn.Embedding(max_desc, r)     
        
        c = 5 * r 
        
        self.W1 = nn.Linear(c, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc1 = nn.Linear(hidden_dim, 16)
        self.fc2 = nn.Linear(16, 1) 
        self.relu = nn.ReLU()

    def forward(self, features, A):
        # Trích xuất vector cho từng đặc trưng (Kích thước [N, 5])
        e_name = self.emb_name(features[:, 0])
        e_left = self.emb_left(features[:, 1])
        e_right = self.emb_right(features[:, 2])
        e_height = self.emb_height(features[:, 3])
        e_desc = self.emb_desc(features[:, 4])
        
        # Nối (Concatenate) theo đúng công thức của bài báo (Kích thước [N, c])
        x = torch.cat([e_name, e_left, e_right, e_height, e_desc], dim=1)
        
        x = torch.matmul(A, x)
        x = self.relu(self.W1(x))
        x = torch.matmul(A, x)
        x = self.relu(self.W2(x))
        
        graph_embed = torch.mean(x, dim=0)
        
        out = self.relu(self.fc1(graph_embed))
        out = torch.sigmoid(self.fc2(out)) 
        return out

gnn_surrogate = Exact_MTGCN_Model(NUM_NODE_TYPES).to(DEVICE)
optimizer = optim.Adam(gnn_surrogate.parameters(), lr=GNN_LR)
criterion = nn.MSELoss()

# ==========================================
# 5. CÁC TOÁN TỬ TIẾN HÓA VÀ CHỌN LỌC
# ==========================================
toolbox.register("select_tournament", tools.selTournament, tournsize=5)

def select_lexicase(individuals, k):
    selected = []
    for _ in range(k):
        candidates = individuals[:]
        cases = list(range(len(candidates[0].fold_accuracies)))
        random.shuffle(cases) 
        for case in cases:
            max_val = max(ind.fold_accuracies[case] for ind in candidates)
            candidates = [ind for ind in candidates if ind.fold_accuracies[case] == max_val]
            if len(candidates) == 1: break
        selected.append(random.choice(candidates))
    return selected

def cxMultiTree_wrapper(ind1, ind2):
    fit1 = ind1.fitness.values[0] if ind1.fitness.valid else 0
    fit2 = ind2.fitness.values[0] if ind2.fitness.valid else 0
    p_fit = [max(fit1, fit2), min(fit1, fit2), (fit1+fit2)/2.0]
    
    idx = random.randint(0, NUM_TREES-1)
    ind1[idx], ind2[idx] = gp.cxOnePoint(ind1[idx], ind2[idx])
    ind1.parent_fitness, ind2.parent_fitness = p_fit, p_fit
    return ind1, ind2

def mutMultiTree_wrapper(individual):
    fit = individual.fitness.values[0] if individual.fitness.valid else 0
    p_fit = [fit, fit, fit]
    
    idx = random.randint(0, NUM_TREES-1)
    mutant = gp.mutUniform(individual[idx], toolbox.expr, pset)
    individual[idx] = mutant[0]
    individual.parent_fitness = p_fit
    return individual,

toolbox.register("mate", cxMultiTree_wrapper)
toolbox.register("mutate", mutMultiTree_wrapper)

# ==========================================
# 6. VÒNG LẶP TIẾN HÓA MTGCN (MAIN)
# ==========================================
def main(X_train_raw, y_train_raw, X_test_raw, y_test_raw):
    start_time_total = time.time()
    n_jobs = cpu_count()
    
    print(f"\n[HỆ THỐNG] EXACT MTGCN Surrogate - Tỷ lệ đánh giá thực tế: {P_RATIO*100}%")
    print(f"[HỆ THỐNG] Đa luồng: {n_jobs} nhân CPU cho Đánh giá Thực tế\n")

    pop = toolbox.population(n=POP_SIZE)
    Archive_Graphs = []  
    Archive_Fits = []
    Table = {}    
    E_evals = 0
    E_max = POP_SIZE + MAX_GEN * int(P_RATIO * POP_SIZE)

    # Thế hệ 0
    tasks = [(ind, X_train_raw, y_train_raw, pset) for ind in pop]
    results = Parallel(n_jobs=n_jobs)(delayed(evaluate_wrapper)(task) for task in tasks)
    
    for ind, res in zip(pop, results):
        ind.fitness.values = (res[0],)     
        ind.fold_accuracies = res[1] 
        ind.parent_fitness = [res[0], res[0], res[0]]
        
        # Gọi hàm extract mới
        features_matrix, A_mat = extract_mtgcn_multitree_exact(ind)
        Archive_Graphs.append((torch.tensor(features_matrix, dtype=torch.long), 
                               torch.tensor(A_mat, dtype=torch.float32)))
        Archive_Fits.append(res[0])
        
        Table[get_ind_hash(ind)] = (res[0], res[1])
        E_evals += 1

    best_acc_global = max(res[0] for res in results)

    for gen in range(MAX_GEN):
        start_time_gen = time.time()
        print(f"-- Thế hệ {gen+1}/{MAX_GEN} (Evals: {E_evals}/{E_max}) --", flush=True)
        
        elites = tools.selBest(pop, k=ELITISM)
        num_select = len(pop) - ELITISM

        if gen < (MAX_GEN // 2):
            offspring = toolbox.select_tournament(pop, num_select)
        else:
            offspring = select_lexicase(pop, num_select)
            if gen == (MAX_GEN // 2): print("   -> Đã chuyển sang Lexicase Selection")
            
        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        to_predict = []
        
        for ind in invalid_ind:
            h = get_ind_hash(ind)
            if h in Table:
                ind.fitness.values = (Table[h][0],)
                ind.fold_accuracies = Table[h][1]
            else:
                to_predict.append(ind)
                
        # Áp dụng EXACT MTGCN Surrogate
        if to_predict:
            gnn_surrogate.eval()
            predicted_scores = []
            
            with torch.no_grad():
                for ind in to_predict:
                    features_matrix, A_mat = extract_mtgcn_multitree_exact(ind)
                    t_nodes = torch.tensor(features_matrix, dtype=torch.long).to(DEVICE)
                    t_A = torch.tensor(A_mat, dtype=torch.float32).to(DEVICE)
                    
                    pred_fit = gnn_surrogate(t_nodes, t_A).item()
                    predicted_scores.append(pred_fit)
            
            for ind, pred_fit in zip(to_predict, predicted_scores):
                ind.fitness.values = (pred_fit,)
                ind.fold_accuracies = [pred_fit] * 5 
            
            p_size = max(1, int(P_RATIO * POP_SIZE))
            to_predict.sort(key=lambda x: x.fitness.values[0], reverse=True)
            top_p_inds = to_predict[:p_size]
            
            # Đánh giá thực tế các cá thể ưu tú nhất
            tasks_gen = [(ind, X_train_raw, y_train_raw, pset) for ind in top_p_inds]
            results_gen = Parallel(n_jobs=n_jobs)(delayed(evaluate_wrapper)(task) for task in tasks_gen)
            
            for ind, res in zip(top_p_inds, results_gen):
                ind.fitness.values = (res[0],)
                ind.fold_accuracies = res[1] 
                
                n_type, A_mat = extract_mtgcn_multitree_exact(ind)
                Archive_Graphs.append((torch.tensor(n_type, dtype=torch.long), 
                                       torch.tensor(A_mat, dtype=torch.float32)))
                Archive_Fits.append(res[0])
                Table[get_ind_hash(ind)] = (res[0], res[1])
                E_evals += 1
                
            # Cắt giảm Archive để tránh quá tải bộ nhớ
            if len(Archive_Graphs) > MAX_ARCHIVE_SIZE:
                Archive_Graphs = Archive_Graphs[-MAX_ARCHIVE_SIZE:]
                Archive_Fits = Archive_Fits[-MAX_ARCHIVE_SIZE:]

            # Huấn luyện lại EXACT MTGCN
            gnn_surrogate.train()
            total_loss = 0
            for _ in range(GNN_EPOCHS):
                epoch_loss = 0
                for (t_nodes, t_A), true_f in zip(Archive_Graphs, Archive_Fits):
                    t_nodes = t_nodes.to(DEVICE)
                    t_A = t_A.to(DEVICE)
                    target = torch.tensor([true_f], dtype=torch.float32).to(DEVICE)
                    
                    optimizer.zero_grad()
                    pred_f = gnn_surrogate(t_nodes, t_A)
                    loss = criterion(pred_f, target)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
                total_loss = epoch_loss / len(Archive_Graphs)

        pop[:] = elites + offspring
        
        best_ind = tools.selBest(pop, 1)[0]
        if best_ind.fitness.values[0] > best_acc_global:
            best_acc_global = best_ind.fitness.values[0]
            
        print(f"   Max Acc (Validation) : {best_ind.fitness.values[0]*100:.2f}%")
        print(f"   Kỷ lục hiện tại      : {best_acc_global*100:.2f}%")
        print(f"   MTGCN Loss           : {total_loss:.4f}" if to_predict else "   MTGCN Loss           : Không update")
        print(f"   [Thời gian]          : {time.time() - start_time_gen:.2f} giây\n", flush=True)

    print(f"\n================ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP ================")
    best_ind = tools.selBest(pop, 1)[0]
    
    X_train_feats = gp_transform_data(best_ind, X_train_raw, pset)
    X_train_feats = np.nan_to_num(X_train_feats, nan=0.0, posinf=LIMIT, neginf=-LIMIT)
    X_test_feats = gp_transform_data(best_ind, X_test_raw, pset)
    X_test_feats = np.nan_to_num(X_test_feats, nan=0.0, posinf=LIMIT, neginf=-LIMIT)
    
    scaler = StandardScaler()
    X_train_feats = scaler.fit_transform(X_train_feats)
    X_test_feats = scaler.transform(X_test_feats)
    
    clf = SVC(kernel='linear', max_iter=2000, random_state=42)
    clf.fit(X_train_feats, y_train_raw)
    test_acc = np.mean(clf.predict(X_test_feats) == y_test_raw)
    
    total_time = time.time() - start_time_total
    hours, rem = divmod(total_time, 3600)
    minutes, seconds = divmod(rem, 60)
    
    print("-" * 50)
    print(f"Độ chính xác Validation (Cao nhất) : {best_ind.fitness.values[0] * 100:.2f}%")
    print(f"ĐỘ CHÍNH XÁC TEST (BÁO CÁO CUỐI)   : {test_acc * 100:.2f}%")
    print(f"TỔNG THỜI GIAN CHẠY THUẬT TOÁN     : {int(hours)} giờ {int(minutes)} phút {seconds:.2f} giây")
    print("==================================================================")

if __name__ == "__main__":
    if not os.path.exists(THU_MUC_MOI):
        prepare_train_test_dataset(THU_MUC_GOC, THU_MUC_MOI, num_train=5, img_size=(50, 50))
    else:
        print(f"Thư mục {THU_MUC_MOI} đã tồn tại. Bỏ qua bước tiền xử lý.")

    print("   --- ĐANG TẢI TẬP TRAIN ---  ")
    X_train, y_train = load_dataset_raw(DATA_DIR_TRAIN)
    print(f"Kích thước tập Train: {X_train.shape}")
    
    print("   --- ĐANG TẢI TẬP TEST ---  ")
    X_test, y_test = load_dataset_raw(DATA_DIR_TEST)
    print(f"Kích thước tập Test: {X_test.shape}")

    if len(X_train) > 0 and len(X_test) > 0:
        main(X_train, y_train, X_test, y_test)
    else:
        print("Lỗi: Không tìm thấy dữ liệu!")

[HỆ THỐNG] EXACT MTGCN Surrogate
[HỆ THỐNG] PyTorch đang sử dụng thiết bị: cpu
Bắt đầu chia tập Train/Test (Train: 5 ảnh/class, Resize: (50, 50))...
Hoàn thành! Tổng Train: 155 ảnh | Tổng Test: 640 ảnh

   --- ĐANG TẢI TẬP TRAIN ---  
Kích thước tập Train: (155, 50, 50, 3)
   --- ĐANG TẢI TẬP TEST ---  
Kích thước tập Test: (640, 50, 50, 3)

[HỆ THỐNG] EXACT MTGCN Surrogate - Tỷ lệ đánh giá thực tế: 20.0%
[HỆ THỐNG] Đa luồng: 4 nhân CPU cho Đánh giá Thực tế

-- Thế hệ 1/50 (Evals: 100/1100) --
   Max Acc (Validation) : 50.36%
   Kỷ lục hiện tại      : 50.36%
   MTGCN Loss           : 0.0003
   [Thời gian]          : 26.15 giây

-- Thế hệ 2/50 (Evals: 120/1100) --
   Max Acc (Validation) : 53.55%
   Kỷ lục hiện tại      : 53.55%
   MTGCN Loss           : 0.0001
   [Thời gian]          : 30.35 giây

-- Thế hệ 3/50 (Evals: 140/1100) --
   Max Acc (Validation) : 53.55%
   Kỷ lục hiện tại      : 53.55%
   MTGCN Loss           : 0.0001
   [Thời gian]          : 30.22 giây

-- Thế hệ 4/50 (Ev